# Marker Repo - submit lists

This notebook allows you to create and upload a marker list. <br>
<br>
A marker list consists of two components: the metadata and the list(s) of markers. <br>
The metadata is entered manually, while the markers are taken from a tab delimited file with one or two columns. <br>
<br>
In the settings you only have to specify the path to this marker file and information about the column(s) of the file <b>(2)</b>. <br>
<br>
The input of the metadata as well as the verification and extension of the markers themselves, is supported by whitelists. The whitelists are located in an external repository, which is downloaded or updated before the data is entered <b>(3)</b>. <br>
<br>
Now the first metadata can be entered: the name of the list, the organism, and the type of markers (genes or genomic regions) <b>(4)</b>. <br>
<br>
At this point the marker file is being read in. If the markers are genes, they will be filtered and expanded using whitelists (Gene name and Ensembl ID) <b>(5)</b>. <br>
<br>
Finally, the remaining metadata can be entered and the new marker list can be saved <b>(6)</b>. <br>
<br>
After a last validation <b>(7)</b>, the list can be published - if desired - and added to the repository <b>(8)</b>.

## 1. Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.update_uids as u_uids
import markerrepo.generate_metafile as gm
import markerrepo.validate_yaml as validate
import markerrepo.utils as utils

%load_ext autoreload
%autoreload 2

## 2. Settings

Specify path of the cloned repository.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"

The path of the list to be added to the marker repo. <br>
The file must consist of one marker per line or two tab separated columns (marker and info like cell type).

In [ ]:
LIST_PATH = '/mnt/workspace/mkessle/projects/annotate_by_marker_and_features/mouse_rat_hackathon'

Additional information of the columns. <br>
marker_col: The column where the markers are stored <br>
info_col: The column where the information of the markers are stored (e.g. cell type, phase, ...). <br>
0 - first column, 1 - second column

If there are only markers available (one column), enter a string with the description of the markers into the info_col parameter - e.g. info_col = "mitochondrial".

In [ ]:
marker_col = 0
info_col = 1

## 3. Get whitelists

Pull whitelist repository and update if necessary.

In [ ]:
mr.get_whitelists()

## 4. Get essential metadata

Enter essential metadata: Liste name, organism and marker type

In [ ]:
LIST_NAME = input("Please enter the name of the marker list: ")
ORGANISM = mr.select(key="organism")
MARKER_TYPE = mr.select(key="marker_type")

Read genes whitelist in order to filter and extend marker genes

In [ ]:
if MARKER_TYPE == "Genes":
    gene_dict = mr.get_gene_dict(ORGANISM)

## 5. Transform marker list

<b>Read</b>, <b>filter</b>, <b>extend</b> and <b>convert</b> marker list in order to append it to the yaml file.

In [ ]:
markers = mr.get_list(LIST_PATH, info_col=info_col, marker_col=marker_col).drop_duplicates()
markers['Marker'] = markers['Marker'].str.upper()
print("All markers of provided list:")
display(markers)

if MARKER_TYPE == "Genes":
    markers_removed = markers[~markers['Marker'].isin(gene_dict.keys())]
    print("Removed markers:")
    display(markers_removed)
    markers_filtered = markers[markers['Marker'].isin(gene_dict.keys())]
    markers_extended = mr.update_markers(markers_filtered, gene_dict)
    print("Filtered and extended markers:")
    display(markers_extended)
    marker_dict = mr.dataframe_to_dict(markers_extended)
else:
    marker_dict = mr.dataframe_to_dict(markers)
    
marker_list = []
for name in marker_dict.keys():
    marker_list.append({'name': name, 'markers': marker_dict[name]})

## 6. Enter metadata

Enter general metadata, tags and add marker list(s) automatically.

In [ ]:
UID = mr.get_uid()
file_name = f"{LIST_NAME}_{UID}.yaml"

list_path = gm.generate_file(UID, LIST_NAME, False, marker_list, ORGANISM, MARKER_TYPE)

## 7. Validation

Check whether the format of the yaml file is correct.

In [ ]:
if validate.validate_file(utils.read_in_yaml(f"{file_name}")):
    print(f"No errors were found concerning the '{LIST_NAME}' marker list.")

## 8. Push list to repository

Pulls the latest changes, creates a new branch with the given list name, adds the new list, commits the changes and pushes the new branch to the remote repository.

In [ ]:
mr.push_marker_list(list_path)